In [5]:
import numpy as np
import pandas as pd

In [ ]:
def zfunc(beta: np.ndarray, x: np.ndarray) -> float:
    """
    Tính z = beta.T @ x
    Trong đó:
    - beta: vector hệ số hồi quy (shape: (k+1,))
    - x: vector đặc trưng của 1 mẫu (shape: (k+1,))
    """
    return float(np.dot(beta, x))

def sigmoid(z: float) -> float:
    """
    Hàm sigmoid: sigma(z) = 1 / (1 + exp(-z))
    """
    return 1 / (1 + np.exp(-z))

def predict_proba(beta: np.ndarray, x: np.ndarray) -> float:
    """
    Tính xác suất dự đoán của mô hình logistic regression cho 1 mẫu x
    - beta: vector hệ số hồi quy
    - x: vector đặc trưng của 1 mẫu
    """
    z = zfunc(beta, x)
    return sigmoid(z)

def predict_all(beta: np.ndarray, X: np.ndarray) -> np.ndarray:
    """
    Trả về vector xác suất dự đoán cho toàn bộ tập X (n mẫu)
    - beta: (k+1,)
    - X: ma trận đặc trưng (n, k+1)
    """
    z = X @ beta
    return sigmoid(z)

def compute_loss(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
    """
    Tính hàm mất mát logistic loss (cross-entropy)
    
    Parameters:
    - beta: vector hệ số (k+1,)
    - X: ma trận đặc trưng (n, k+1)
    - y: vector nhãn thật (n,)
    
    Returns:
    - loss: scalar
    """
    eps = 1e-15  # để tránh log(0)
    y_hat = sigmoid(X @ beta)
    y_hat = np.clip(y_hat, eps, 1 - eps)  # giới hạn tránh log(0)
    n = X.shape[0]
    loss = - (1 / n) * np.sum(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
    return loss

def compute_gradient(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Tính gradient của hàm mất mát logistic:
    ∇L(β) = (1/n) * sum_i [(ŷ_i - y_i) * x_i]
    
    Parameters:
    - beta: vector hệ số (k+1,)
    - X: ma trận đặc trưng (n, k+1)
    - y: vector nhãn thật (n,)
    
    Returns:
    - gradient: vector gradient (k+1,)
    """
    n = X.shape[0]
    y_hat = sigmoid(X @ beta)  # shape (n,)
    grad = (1 / n) * X.T @ (y_hat - y)
    return grad

def gradient_norm(grad: np.ndarray) -> float:
    """
    Tính chuẩn L2 (Euclidean norm) của gradient
    
    Parameters:
    - grad: vector gradient (k+1,)
    
    Returns:
    - norm: scalar
    """
    return np.linalg.norm(grad)

In [44]:
def train_logistic_regression(
    X: np.ndarray,
    y: np.ndarray,
    alpha: float = 0.01,
    iterations: int = 3
) -> np.ndarray:
    """
    Huấn luyện logistic regression bằng Gradient Descent

    Parameters:
    - X: ma trận đặc trưng (n, k+1)
    - y: vector nhãn (n,)
    - alpha: learning rate
    - iterations: số lần lặp

    Returns:
    - beta: vector hệ số (k+1,)
    """
    k = X.shape[1]
    beta = np.zeros(k)  # Khởi tạo beta = 0

    for t in range(1, iterations + 1):
        grad = compute_gradient(beta, X, y)
        loss = compute_loss(beta, X, y)
        norm = gradient_norm(grad)

        print(f"🔁 Iteration {t}")
        print(f"  ||∇L(β)|| = {norm:.6f}")
        print(f"  L(β) = {loss:.6f}")
        print(f"  β = {beta}")
        print("-" * 40)

        beta -= alpha * grad

    return beta


In [ ]:
def scale_features(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Chuẩn hóa các cột của X theo z-score:
        X_scaled = (X - mean) / std

    Parameters:
    - X: ma trận đặc trưng (n, k)

    Returns:
    - X_scaled: ma trận đã chuẩn hóa (n, k)
    - mean: vector trung bình theo cột (k,)
    - std: vector độ lệch chuẩn theo cột (k,)
    """
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0, ddof=0)
    X_scaled = (X - mean) / std
    return X_scaled, mean, std


In [ ]:
# Đọc dữ liệu từ Excel
file_path = "Team11_HW6.xlsx"
df_train = pd.read_excel(file_path, sheet_name="Data-LR-Short-Train", usecols="A:D")
df_test = pd.read_excel(file_path, sheet_name="Data-LR-Short-Test")

In [18]:
print(df_train.tail())
print("=================================================")
print(df_test.tail())

    income   amount  down_payment  label_refused
25  157500  79443.0        7947.0              0
26  180000  87259.5        8730.0              0
27  180000  47927.7        4795.2              0
28  108000  44973.9        3600.9              1
29  103500  88110.0       18000.0              0
   income    amount  down_payment  label_refused
5  202500   49495.5        9900.0              0
6  180000   71923.5        7195.5              0
7  121500  108000.0       10800.0              1
8  202500   62050.5       22500.0              1
9  225000   22126.5        4500.0              1


In [ ]:
# Tách X và y
X_train = df_train[["income", "amount", "down_payment"]].copy()
y_train = df_train["label_refused"].values
X_test = df_test[["income", "amount", "down_payment"]].copy()
y_test = df_test["label_refused"].values

In [ ]:
# Tính mean và variance theo cột
mean_train = X_train.mean()
var_train = X_train.var(ddof=0)  # ddof=0: phương sai tổng thể (population variance)

In [ ]:
print("Mean:\n", mean_train)
print("Variance:\n", var_train)

Mean:
 income          183150.000
amount           64010.517
down_payment      8901.567
dtype: float64
Variance:
 income          8.961502e+09
amount          1.470512e+09
down_payment    1.528948e+08
dtype: float64


In [33]:
# Scale X
X_train_scaled = (X_train - mean_train) / np.sqrt(var_train)
X_test_scaled = (X_test - mean_train) / np.sqrt(var_train)

In [26]:
print(X_train_scaled.tail())
print("=================================================")
print(X_test_scaled.tail())

      income    amount  down_payment
25 -0.270955  0.402440     -0.077199
26 -0.033275  0.606275     -0.013875
27 -0.033275 -0.419399     -0.332094
28 -0.793850 -0.496427     -0.428681
29 -0.841386  0.628454      0.735818
     income    amount  down_payment
5  0.204405 -0.378515      0.080746
6 -0.033275  0.206351     -0.137975
7 -0.651242  1.147135      0.153532
8  0.204405 -0.051112      1.099746
9  0.442084 -1.092230     -0.355968


In [35]:
## thêm 1 vào matrix
X_input_train = np.hstack([np.ones((X_train_scaled.shape[0], 1)), X_train_scaled.values])  # shape (n, k+1)
X_input_test = np.hstack([np.ones((X_test_scaled.shape[0], 1)), X_test_scaled.values])


In [ ]:
print(X_input_train)
print("=================================================")
print(X_input_test)

In [45]:
final_beta = train_logistic_regression(X_input_train, y_train, alpha=0.01, iterations=3)

🔁 Iteration 1
  ||∇L(β)|| = 0.084127
  L(β) = 0.693147
  β = [0. 0. 0. 0.]
----------------------------------------
🔁 Iteration 2
  ||∇L(β)|| = 0.083859
  L(β) = 0.693077
  β = [0.         0.00032483 0.00016303 0.00075871]
----------------------------------------
🔁 Iteration 3
  ||∇L(β)|| = 0.083592
  L(β) = 0.693006
  β = [5.10434472e-13 6.48704989e-04 3.24242622e-04 1.51524127e-03]
----------------------------------------
